In [1]:
# 设置环境变量
%env HF_ENDPOINT=https://hf-mirror.com
%env HF_HOME=/root/autodl-tmp/hf

env: HF_ENDPOINT=https://hf-mirror.com
env: HF_HOME=/root/autodl-tmp/hf


In [2]:
# 加载model和tkennizer
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch
from peft import PeftModel

# 模型名称
model_name = "Qwen/Qwen3-8B"
# LoRA微调之后的模型Adapter
lora_adpter = "/root/autodl-tmp/sft/Qwen3-8B/QLoRA/best"

# 基础模型
base_model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16)
# 增加LoRA微调adpater
peft_model = PeftModel.from_pretrained(base_model, lora_adpter, dtype=torch.float16)
# 模型合并
mergered_model = peft_model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(model_name)


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [7]:
# prepare the model input
prompt = "抽取出文本中的关键词：\n标题：拼豆玩具存在安全隐患\n文本：今年3月，贵州一名女孩玩拼豆时，在熨烫环节触电不幸离世。多地监管检查发现，多款拼豆配套加热设备接220V市电，远超儿童玩具安全电压，且多为三无产品，存在触电、起火等严重安全隐患。."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
print(text)

<|im_start|>user
抽取出文本中的关键词：
标题：拼豆玩具存在安全隐患
文本：今年3月，贵州一名女孩玩拼豆时，在熨烫环节触电不幸离世。多地监管检查发现，多款拼豆配套加热设备接220V市电，远超儿童玩具安全电压，且多为三无产品，存在触电、起火等严重安全隐患。.<|im_end|>
<|im_start|>assistant
<think>

</think>




In [10]:
# 模型输入
model_inputs = tokenizer([text], return_tensors="pt").to(mergered_model.device)

# conduct text completion
generated_ids = mergered_model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

# 微调前的输出
# content: 关键词：人工神经网络、猕猴桃、种类识别、模式识别、特征参数、果品、介电特性、品质、研究方法。

# 微调后的输出
# content: 猕猴桃;人工神经网络;种类识别;特征参数;介电特性

thinking content: 
content: 拼豆玩具;加热设备;儿童玩具;安全电压;熨烫;起火;检查;贵州;女孩;三无产品
